# Notebook baseline: abandono de clientes (churn)

Este es el **punto de partida** del proyecto: un notebook típico de data science que
funciona, pero que todavía no es reproducible ni reutilizable fuera de aquí.

En la **práctica 1** lo recorrerás de arriba abajo e identificarás:

1. **Entradas** (columnas) y **target**.
2. **Preprocesamiento** (qué transformaciones se aprenden de los datos).
3. **Entrenamiento** y **evaluación**.
4. **Salidas** (métricas, modelo, predicciones).

Después moverás esas piezas a `src/mlops_course/`.

> Ejecuta siempre con *Kernel → Restart Kernel and Run All Cells*: si falla desde un
> kernel limpio, no es reproducible.

## 1. Carga de datos

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEMILLA = 42
df = pd.read_csv("../data/churn.csv")
print(df.shape)
df.head()

Diccionario de datos (ver `data/README.md`): cada fila es un cliente.
`abandona = 1` indica que el cliente se dio de baja.

In [ ]:
df.info()

In [ ]:
df["abandona"].value_counts(normalize=True).round(3)

In [ ]:
df.describe(include="all").T

## 2. Entradas y target

`id_cliente` es un identificador: **no** es una variable predictiva.

In [ ]:
NUMERICAS = ["antiguedad_meses", "cuota_mensual"]
CATEGORICAS = ["tipo_contrato", "tipo_internet", "metodo_pago"]
TARGET = "abandona"

X = df[NUMERICAS + CATEGORICAS]
y = df[TARGET]

## 3. Split reproducible

Primero se separa el **test** (no se toca hasta el final) y luego **validación**.
`stratify` mantiene la proporción de abandonos en cada partición.

In [ ]:
X_resto, X_test, y_resto, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA
)
X_train, X_val, y_train, y_val = train_test_split(
    X_resto, y_resto, test_size=0.2, stratify=y_resto, random_state=SEMILLA
)
print(f"train: {len(X_train)}, validación: {len(X_val)}, test: {len(X_test)}")

## 4. Preprocesamiento + modelo en un único `Pipeline`

El escalado y el one-hot **se ajustan solo con train** al llamar a `fit`: así se evita
fuga de información (*data leakage*) y el mismo objeto sirve para predecir después.

In [ ]:
preprocesamiento = ColumnTransformer(
    [
        ("numericas", StandardScaler(), NUMERICAS),
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
    ]
)
modelo = Pipeline(
    [
        ("preprocesamiento", preprocesamiento),
        ("clasificador", LogisticRegression(C=1.0, max_iter=1000)),
    ]
)
modelo.fit(X_train, y_train)

## 5. Evaluación en validación

In [ ]:
UMBRAL = 0.5
proba_val = modelo.predict_proba(X_val)[:, 1]
pred_val = (proba_val >= UMBRAL).astype(int)
print(f"ROC AUC validación: {roc_auc_score(y_val, proba_val):.3f}")
print(classification_report(y_val, pred_val, digits=3))

**Salida esperada (orientativa):** ROC AUC de validación en torno a 0,83–0,85.
Pequeñas diferencias entre equipos son normales.

## 6. Predicción para clientes nuevos

In [ ]:
nuevos = pd.read_csv("../data/churn_nuevos.csv")
nuevos["probabilidad"] = modelo.predict_proba(nuevos[NUMERICAS + CATEGORICAS])[:, 1]
nuevos["abandona"] = (nuevos["probabilidad"] >= UMBRAL).astype(int)
nuevos.head()

## Preguntas para la práctica 1

- ¿Qué valores están *hardcodeados* (rutas, semilla, umbral, hiperparámetros)?
  ¿Dónde deberían vivir?
- ¿Qué pasaría si mañana alguien quiere predecir sin abrir este notebook?
- ¿Qué parte se aprende de los datos y qué parte es configuración?
- ¿Dónde se guarda el modelo entrenado? (pista: en ningún sitio, todavía)